# Phase 2 — Claim Extraction, Verdict, /verify, and Orchestration

This notebook covers all four pieces of **Phase 2**: turning raw report text into
a list of checkable claims, verifying each one against the Phase 1 knowledge
base, wiring both into `POST /verify`, and the Airflow DAG that keeps the
knowledge base fresh on a schedule. Phase 2 is done — see
[docs/phase-2-rag-pipeline.md](../docs/phase-2-rag-pipeline.md) for the full
design writeup.

```mermaid
flowchart TB
    subgraph Verify["Verify pipeline — POST /verify"]
        direction TB
        TEXT[Report text] --> LLM[Claim Extractor LLM]
        LLM --> CLAIMS["Claims: entity, metric, value, date"]
        CLAIMS --> RAG[hybrid_search retrieval]
        RAG --> JUDGE[LLM-as-judge]
        JUDGE --> VERDICT["Verdict: VERIFIED / REFUTED / INSUFFICIENT + quote"]
        VERDICT --> API[POST /verify]
    end

    subgraph Orchestration["Airflow DAG — @daily"]
        direction LR
        FETCH["4x fetch_* tasks\n(wikipedia/worldbank/fred/secedgar)"] --> REBUILD[rebuild_vector_store]
    end

    REBUILD -.->|keeps fresh| RAG

    classDef llm fill:#e0ccff,stroke:#7d3cff,color:#2a0a5e
    classDef store fill:#cce5ff,stroke:#0066cc,color:#00264d
    classDef io fill:#d4f7d4,stroke:#2e7d32,color:#1b4d1b
    class LLM,JUDGE llm
    class RAG,REBUILD store
    class TEXT,API io
```


In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "..")

## Claim extractor — structured output, not free text

`src/claim_extractor.py` asks the LLM for a typed `ClaimList` — not free text to parse with regex. Each `Claim` has five fields: `text` (the exact sentence), `entity`, `metric`, `value`, `date`. Each field can be checked against a source later.

The system prompt tells the model to skip opinions and forward-looking statements. Only claims with an entity and a number survive.


In [2]:
from src.claim_extractor import extract_claims

### Example: a snippet from a business report

Three numeric claims — Apple revenue, Apple net income, Walmart revenue — mixed with two opinion sentences that shouldn't survive extraction.


In [3]:
report_text = """Apple Inc. delivered another strong year. Revenue for fiscal year ending 2025-09-27 reached $416,161,000,000, while net income came in at $112,010,000,000.
Management remains optimistic about the upcoming product cycle. Separately, Walmart reported revenue of $713,163,000,000 for fiscal year ending 2026-01-31. 
Analysts believe the retail sector will keep growing."""

print(report_text)

Apple Inc. delivered another strong year. Revenue for fiscal year ending 2025-09-27 reached $416,161,000,000, while net income came in at $112,010,000,000.
Management remains optimistic about the upcoming product cycle. Separately, Walmart reported revenue of $713,163,000,000 for fiscal year ending 2026-01-31. 
Analysts believe the retail sector will keep growing.


In [4]:
claims = extract_claims(report_text)
for c in claims:
    print(c)

text='Revenue for fiscal year ending 2025-09-27 reached $416,161,000,000' entity='Apple Inc.' metric='Revenue' value=416161000000.0 date='2025-09-27'
text='net income came in at $112,010,000,000' entity='Apple Inc.' metric='Net income' value=112010000000.0 date='2025-09-27'
text='Walmart reported revenue of $713,163,000,000 for fiscal year ending 2026-01-31' entity='Walmart' metric='Revenue' value=713163000000.0 date='2026-01-31'


Both opinion sentences ("Management remains optimistic…", "Analysts believe…") were dropped correctly — no entity or number attached. All three claims have clean `entity` / `metric` / `value` / `date` fields, ready to check against the Phase 1 knowledge base.

**Rough edge:** the first two claims share the same `text`. The model split one sentence with two numbers ("revenue… while net income…") into two claims, but reused the whole sentence as `text` for both instead of the specific clause. The structured fields are still correct — this only matters if per-claim quoting matters later.


## Model choice: OpenRouter default, Ollama and Groq as alternatives

Default is OpenRouter's free tier (`LLM_PROVIDER=openrouter`). `src/config.py` currently points it at `poolside/laguna-xs-2.1:free` — $0 cost, fits a capstone with no LLM budget. This default has changed before and may change again if a free model degrades — check `LLM_PROVIDERS` in `src/config.py` for what's live now, don't trust a model name written here.

Two earlier defaults were dropped, both for provider-side reasons, not bugs in this code:

- `nvidia/nemotron-3-ultra-550b-a55b:free` — every call returned `DEGRADED function cannot be invoked` (confirmed with a bare `openai` client too).
- `tencent/hy3:free` — worked fine, later swapped out for the current default.

`src/config.py` also supports `LLM_PROVIDER=ollama` — a local Ollama model (`ornith:latest`) via its OpenAI-compatible endpoint (`http://localhost:11434/v1`, dummy API key). No API key, no rate limits, works offline. Good as a fallback when the OpenRouter free tier is degraded or rate-limited. Not the default — it's CPU-only here (~20s per call once warm, 2+ minutes cold start).

A third option, `LLM_PROVIDER=groq` (`qwen/qwen3.6-27b`), is fast with a generous free tier — compared side by side with the other two below, after the verdict / `/verify` examples.


In [5]:
from langchain_openai import ChatOpenAI
from src.claim_extractor import ClaimList, SYSTEM_PROMPT

ollama_llm = ChatOpenAI(
    model="ornith:latest",
    api_key="ollama",
    base_url="http://localhost:11434/v1",
    temperature=0,
).with_structured_output(ClaimList)

result = ollama_llm.invoke([("system", SYSTEM_PROMPT), ("user", report_text)])
for c in result.claims:
    print(c)

text='Revenue for fiscal year ending 2025-09-27 reached $416,161,000,000' entity='Apple Inc.' metric='revenue' value=416161000000.0 date='fiscal year ending 2025-09-27'
text='net income came in at $112,010,000,000' entity='Apple Inc.' metric='net income' value=112010000000.0 date='fiscal year ending 2025-09-27'
text='Walmart reported revenue of $713,163,000,000 for fiscal year ending 2026-01-31' entity='Walmart' metric='revenue' value=713163000000.0 date='fiscal year ending 2026-01-31'


Same three claims, correctly split — and this time each `text` is the exact clause ("Revenue for…" vs "net income came in at…"), instead of the one shared full sentence OpenRouter's default model produced above. Reminder: "free tier" and "good enough" aren't the same thing. Worth A/B-testing more than one model before locking in a default.


## Verdict — claim + retrieved evidence → VERIFIED / REFUTED / INSUFFICIENT

`src/verifier.py`'s `verify_claim(claim, prompt=VERDICT_PROMPT_V1)` does two things:

1. Embeds the claim text and runs `hybrid_search` (RRF over pgvector + Postgres full-text, from Phase 1/3) against the knowledge base.
2. Passes the claim and the top-5 retrieved chunks to an LLM judge, which returns a structured `Verdict` (`verdict`, `source`, `quote`).

`prompt` is a plain function argument, not hardcoded. Phase 3 can call `verify_claim` with a second prompt on the same claims and compare accuracy — the "LLM evaluation, 2 prompts" line in the capstone rubric.


In [6]:
from src.claim_extractor import Claim
from src.verifier import verify_claim

### Three cases, one from each `data/eval_claims.csv` bucket

- A `VERIFIED` claim (Apple revenue, correct value)
- A `REFUTED` claim (Apple revenue, wrong value — same source chunk as above)
- An `INSUFFICIENT` claim (Netflix — not in the knowledge base at all)


In [7]:
cases = [
    Claim(text="Apple's revenue for fiscal year ending 2025-09-27 was $416,161,000,000.",
          entity="Apple", metric="Revenue", value=416161000000.0, date="2025-09-27"),
    Claim(text="Apple's revenue for fiscal year ending 2025-09-27 was $250,000,000,000.",
          entity="Apple", metric="Revenue", value=250000000000.0, date="2025-09-27"),
    Claim(text="Netflix's revenue for fiscal year 2025 was $39 billion.",
          entity="Netflix", metric="Revenue", value=39000000000.0, date="2025"),
]

for claim in cases:
    v = verify_claim(claim)
    print(claim.text, "->", v.verdict, "|", v.source, "|", (v.quote or "")[:80])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Apple's revenue for fiscal year ending 2025-09-27 was $416,161,000,000. -> VERIFIED | Apple Inc. (AAPL) — Revenue | Apple Inc. (AAPL) reported Revenue of $416,161,000,000 for fiscal year ending 20
Apple's revenue for fiscal year ending 2025-09-27 was $250,000,000,000. -> REFUTED | Apple Inc. (AAPL) — Revenue | Apple Inc. (AAPL) reported Revenue of $416,161,000,000 for fiscal year ending 20
Netflix's revenue for fiscal year 2025 was $39 billion. -> INSUFFICIENT | None | 


All three verdicts are correct:

- **VERIFIED** — matched the exact chunk, same value.
- **REFUTED** — retrieval pulled the *same* Apple revenue chunk (closest match by entity + metric). The judge compared values and caught the mismatch, instead of just confirming "some Apple revenue chunk exists."
- **INSUFFICIENT** — Netflix isn't in the knowledge base (`TICKERS` in `fetch_secedgar.py` covers only 8 companies). Retrieval still returns its top-5 nearest chunks regardless. The judge had to recognize that none of them cover Netflix, instead of defaulting to whichever ranked first — the prompt instruction doing its job, not an empty-results shortcut in the code.


## `POST /verify` — both pieces wired into one endpoint

`src/api.py` chains `extract_claims` → `verify_claim` per claim and returns a list of verdicts. This demo calls it in-process via FastAPI's `TestClient` — no need to run a separate `uvicorn` process.


In [10]:
from fastapi.testclient import TestClient
from src.api import app

client = TestClient(app)

In [12]:
import json

resp = client.post("/verify", json={"text": "Apple reported revenue of $416,161,000,000 for fiscal year ending 2025-09-27."})
print(resp.status_code)
print(json.dumps(resp.json(), indent=2))

200
{
  "claims": [
    {
      "claim": "Apple reported revenue of $416,161,000,000 for fiscal year ending 2025-09-27.",
      "verdict": "VERIFIED",
      "source": "Apple Inc. (AAPL) \u2014 Revenue",
      "quote": "Apple Inc. (AAPL) reported Revenue of $416,161,000,000 for fiscal year ending 2025-09-27 (10-K filed 2025-10-31)."
    }
  ]
}


**On free-tier latency:** this call took under a minute end to end — free-tier providers aren't latency-consistent. `src/llm.py` sets an explicit per-call `timeout` (with one retry), so a stuck request fails fast instead of hanging on the `openai` client's 600s default.


## Comparing providers on this notebook's use cases

`src/config.py` configures three providers: OpenRouter (default), Groq, and Ollama.

Don't pick one and hope. Test all three, on the same three use cases used above:

- a plain call
- a single-object structured extraction
- a multi-item structured extraction

Same code as the rest of this project: plain `ChatOpenAI(...)` and `with_structured_output(...)`. No per-provider tuning.


In [13]:
from src.config import LLM_PROVIDERS

providers_to_test = [
    (name, cfg["default_model"], cfg["base_url"], cfg["api_key"])
    for name, cfg in LLM_PROVIDERS.items()
]
[(name, model) for name, model, _, _ in providers_to_test]

[('openrouter', 'poolside/laguna-xs-2.1:free'),
 ('groq', 'qwen/qwen3.6-27b'),
 ('ollama', 'ornith:latest')]

### Use case 1 — a plain call (course lesson [`07-llm.md`](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-agentic-rag/lessons/07-llm.md) style)

System instructions + a user question, free-text answer. No structure — just checking whether each provider/model responds sensibly at all.


In [14]:
from langchain_openai import ChatOpenAI

for name, model, base_url, api_key in providers_to_test:
    llm = ChatOpenAI(model=model, base_url=base_url, api_key=api_key, temperature=0)
    try:
        answer = llm.invoke([
            ("system", "Answer in one short sentence."),
            ("user", "What is 15% of 240?"),
        ])
        print(f"{name} ({model}): {answer.content[:200]!r}")
    except Exception as e:
        print(f"{name} ({model}): FAILED — {str(e)[:150]}")

openrouter (poolside/laguna-xs-2.1:free): '15% of 240 is 36.'
groq (qwen/qwen3.6-27b): '\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - Question: "What is 15% of 240?"\n   - Constraint: "Answer in one short sentence."\n\n2.  **Calculate the Answer:**\n   - 15% of 240 = '
ollama (ornith:latest): '15% of 240 is 36.'


### Use case 2 — structured output, one object (`Claim`)

Same `Claim` model used in `verifier.py` above: one entity, one value.


In [15]:
for name, model, base_url, api_key in providers_to_test:
    llm = ChatOpenAI(model=model, base_url=base_url, api_key=api_key, temperature=0)
    try:
        result = llm.with_structured_output(Claim).invoke(
            "Apple revenue was $394B in FY2023."
        )
        print(f"{name} ({model}): OK — {result}")
    except Exception as e:
        print(f"{name} ({model}): FAILED — {str(e)[:150]}")

openrouter (poolside/laguna-xs-2.1:free): OK — text="Apple's reported revenue for FY2023 was **$383 billion**, not $394 billion. This figure represents the total revenue for the fiscal year ending September 30, 2023. Here's a detailed breakdown of Apple's FY2023 performance:\n\n### Key Revenue Breakdown:\n1. **iPhone**: $205.5 billion (53.7% of total revenue)\n2. **Services**: $80.3 billion (20.9% of total revenue)\n3. **Mac**: $28.5 billion (7.4% of total revenue)\n4. **iPad**: $23.5 billion (6.1% of total revenue)\n5. **Wearables, Home and Accessories**: $35.3 billion (9.2% of total revenue)\n6. **Other Products**: $9.9 billion (2.6% of total revenue)\n\n### Key Highlights:\n- **Growth Drivers**: Strong iPhone 14 sales, continued expansion of the Services ecosystem (including App Store, iCloud, and Apple Music), and growth in Wearables (e.g., Apple Watch and AirPods).\n- **Challenges**: Supply chain constraints, economic headwinds in key markets, and increased competition in the smar

### Use case 3 — structured output, multiple items (`ClaimList` — the shape `extract_claims()` actually uses)

Same `report_text` from the top of this notebook, three claims to pull out. This generation is longer than use case 2, so worth checking separately — "works for one field" doesn't guarantee "works for a list of them".


In [16]:
from src.claim_extractor import ClaimList, SYSTEM_PROMPT

for name, model, base_url, api_key in providers_to_test:
    llm = ChatOpenAI(model=model, base_url=base_url, api_key=api_key, temperature=0)
    try:
        result = llm.with_structured_output(ClaimList).invoke(
            [("system", SYSTEM_PROMPT), ("user", report_text)]
        )
        print(f"{name} ({model}): OK — {len(result.claims)} claims")
    except Exception as e:
        print(f"{name} ({model}): FAILED — {str(e)[:150]}")

openrouter (poolside/laguna-xs-2.1:free): OK — 3 claims
groq (qwen/qwen3.6-27b): FAILED — Error code: 400 - {'error': {'message': 'This model does not support response format `json_schema`. See supported models at https://console.groq.com/d
ollama (ornith:latest): OK — 3 claims


### Takeaway

OpenRouter and Ollama pass all three cases. Groq (`qwen/qwen3.6-27b`) fails both structured cases — it doesn't support `with_structured_output()`'s default (`json_schema`).

Pick a provider/model that passes all three cleanly. That's a safe default for the real code (`claim_extractor.py`, `verifier.py`) — same `with_structured_output()` call, no changes needed.

If a model fails, swap the model. Don't add per-provider config to the client — that's what `LLM_MODEL` in `.env` is for.


## Orchestration — Airflow DAG for daily ingestion

Phase 1 built the knowledge base once, by hand. In production the sources
change — SEC EDGAR gets new filings, FRED updates macro series daily — so
something has to re-run ingestion on a schedule, or the vector store
answers with stale facts.

**Input:** nothing but the `@daily` schedule trigger + secrets from `.env`
+ whatever's already in the `*_raw` Postgres tables. Each `ingest.fetch_*`
resource is a dlt `write_disposition="merge"` on its own primary key, so
re-running a fetch is idempotent — no duplicate rows pile up.

**Output:** refreshed `wikipedia_raw` / `worldbank_raw` / `fred_raw` /
`secedgar_raw` schemas, then a full truncate-and-rebuild of the derived
`documents` / `document_chunks` vector store that hybrid search reads from.


### Design: orchestration only, no reimplemented logic

`dags/fact_checker_dag.py` imports each ingest module's existing `run()`
function — it does not reimplement fetch/chunk/embed logic. That keeps
single responsibility where it already lived (one module per source, one
module for the rebuild):

```python
from ingest import build_vector_store, fetch_fred, fetch_secedgar, fetch_wikipedia, fetch_worldbank

FETCH_MODULES = {
    "wikipedia": fetch_wikipedia,
    "worldbank": fetch_worldbank,
    "fred": fetch_fred,
    "secedgar": fetch_secedgar,
}

with DAG("fact_checker_daily_ingestion", schedule="@daily", ...) as dag:
    fetch_tasks = [
        PythonOperator(task_id=f"fetch_{name}", python_callable=module.run)
        for name, module in FETCH_MODULES.items()
    ]
    rebuild_vector_store = PythonOperator(
        task_id="rebuild_vector_store", python_callable=build_vector_store.run
    )
    fetch_tasks >> rebuild_vector_store
```

A few principles this leans on:

- **Single responsibility.** Each `fetch_*` task owns exactly one source. `rebuild_vector_store` owns exactly one thing too: turning raw tables into the derived vector store.
- **Open/closed.** Adding a fifth source later means one new `ingest/fetch_*.py` module + one new dict entry above — no existing task changes.
- **Dependency inversion.** The DAG depends on the stable `run()` interface each module already exposes, not on dlt/psycopg2 internals underneath it.
- **Dependency graph matches the real one, not an assumed one.** The four fetch tasks don't depend on each other — they're independent sources. Only `rebuild_vector_store` depends on all four finishing, because `build_vector_store.run()` re-reads every `*_raw` table from scratch on each run (full rebuild, not incremental — see Phase 1 doc).

**Runtime is a separate concern from the app.** The Airflow image (`Dockerfile.airflow`) only installs what the DAG's tasks need to import — dlt, psycopg2, pgvector, sentence-transformers, requests, python-dotenv — not langchain/ragas/fastapi, since orchestration never touches the RAG, eval, or API layers. `docker-compose.yml` runs it as its own `airflow` service (`standalone` mode: one container, SQLite metadata db — enough for a capstone, no second Postgres needed just to hold Airflow's own state).


In [18]:
import subprocess

# Airflow runs in its own container (`docker compose up -d airflow`), not
# in this notebook's kernel — so we shell out to check it instead of importing
# the DAG module directly (that would try to import `airflow` into this env).
result = subprocess.run(
    ["docker", "exec", "fact-checker-airflow", "airflow", "dags", "list"],
    capture_output=True, text=True,
)
print(result.stdout or result.stderr)

dag_id                       | fileloc                               | owners  | is_paused
=============================+=======================================+=========+==========
fact_checker_daily_ingestion | /opt/airflow/dags/fact_checker_dag.py | airflow | False    
                                                                                          



`airflow dags list` (cell above) shows `fact_checker_daily_ingestion` with no import errors. Verified separately, not shown here: `airflow tasks list` lists all 5 tasks, and `airflow tasks test fact_checker_daily_ingestion fetch_wikipedia 2026-07-11` ran end to end — dlt loaded a package into Postgres, task marked `SUCCESS`.


## What's next

Claim extraction, verdict logic, `POST /verify`, and the Airflow DAG are
all working — Phase 2 is done. Next (see
[docs/phase-3-evaluation.md](../docs/phase-3-evaluation.md) and
[notebooks/phase3_evaluation.ipynb](phase3_evaluation.ipynb)): hybrid
search, reranking, and RAGAS evaluation.

[← Back to README](../README.md)
